In [ ]:
##This notebook serves as execution for wrapped SQL queries for exploratory analysis of FAERS dataset before subsequent analysis


# # FAERS Database Schema Overview

# This project uses FDA FAERS (FDA Adverse Event Reporting System) quarterly ASCII datasets
# combined into a single SQLite database.

# Each adverse event report can contain:

# - one demographic record
# - multiple drugs
# - multiple reactions
# - multiple outcomes
# - multiple indications

# The primary relational key across tables is **primaryid**.

# caseid groups multiple versions of a case report.

# ---

# # DEMO — Demographic / Case Information

# Contains **one row per adverse event report version**.

# | Column | Meaning |
# |------|------|
# primaryid | unique report identifier (primary key)
# caseid | case identifier (groups related reports)
# caseversion | version of the case report
# i_f_code | initial or follow-up report indicator
# event_dt | date adverse event occurred
# mfr_dt | date manufacturer received report
# init_fda_dt | date FDA initially received report
# fda_dt | date FDA processed report
# rept_cod | report type code
# auth_num | authority number (regulatory tracking)
# mfr_num | manufacturer report number
# mfr_sndr | manufacturer sending the report
# lit_ref | literature reference if reported in publication
# age | patient age
# age_cod | age unit (years, months, days)
# age_grp | age group classification
# sex | patient sex
# e_sub | expedited submission indicator
# wt | patient weight
# wt_cod | weight unit
# rept_dt | report submission date
# to_mfr | report forwarded to manufacturer indicator
# occp_cod | reporter occupation code
# reporter_country | country of reporter
# occr_country | country where event occurred
# source_quarter | dataset quarter used during ingestion

# ---

# # DRUG — Drug Information

# Contains **one row per drug reported in the case**.

# Multiple drugs may appear in a single adverse event report.

# | Column | Meaning |
# |------|------|
# primaryid | report identifier (links to DEMO)
# caseid | case identifier
# drug_seq | sequence number for drugs in report
# role_cod | role of drug (PS = primary suspect, SS = secondary suspect, etc.)
# drugname | reported drug name
# prod_ai | active ingredient
# val_vbm | validity of verbatim medicine
# route | route of administration
# dose_vbm | verbatim dose description
# cum_dose_chr | cumulative dose character
# cum_dose_unit | cumulative dose unit
# dechal | dechallenge indicator (reaction improved after stopping drug)
# rechal | rechallenge indicator (reaction returned after restarting drug)
# lot_num | drug lot number
# exp_dt | expiration date
# nda_num | NDA application number
# dose_amt | dose amount
# dose_unit | dose unit
# dose_form | formulation
# dose_freq | dosing frequency
# source_quarter | dataset quarter

# ---

# # INDI — Drug Indication

# Contains **the medical condition for which the drug was prescribed**.

# | Column | Meaning |
# |------|------|
# primaryid | report identifier
# caseid | case identifier
# indi_drug_seq | drug sequence number
# indi_pt | MedDRA preferred term for indication
# source_quarter | dataset quarter

# ---

# # OUTC — Patient Outcome

# Contains **reported outcomes of the adverse event**.

# Multiple outcomes may exist for a case.

# | Column | Meaning |
# |------|------|
# primaryid | report identifier
# caseid | case identifier
# outc_cod | outcome code (death, hospitalization, disability, etc.)
# source_quarter | dataset quarter

# ---

# # REAC — Adverse Reactions

# Contains **medical reactions reported for the case**.

# | Column | Meaning |
# |------|------|
# primaryid | report identifier
# caseid | case identifier
# pt | MedDRA preferred term describing the reaction
# drug_rec_act | drug received action indicator
# source_quarter | dataset quarter

# ---

# # RPSR — Report Source

# Identifies **who submitted the report**.

# | Column | Meaning |
# |------|------|
# primaryid | report identifier
# caseid | case identifier
# rpsr_cod | reporter source code (physician, consumer, pharmacist, etc.)
# source_quarter | dataset quarter

# ---

# # THER — Drug Therapy Dates

# Contains **therapy start/end dates and duration**.

# | Column | Meaning |
# |------|------|
# primaryid | report identifier
# caseid | case identifier
# dsg_drug_seq | drug sequence number
# start_dt | therapy start date
# end_dt | therapy end date
# dur | duration of therapy
# dur_cod | duration unit
# source_quarter | dataset quarter

# ---

# # Key Relational Structure

# Most tables join on **primaryid**.

# Example conceptual relationship:

#       DEMO
#        |
#     primaryid
#    /     |     \
# DRUG   REAC   OUTC
#   |
#  INDI
#   |
#  THER





#Set project root variables
from pathlib import Path
import sqlite3
import pandas as pd

project_root = Path.cwd().parents[0]
db_path = project_root / "database" / "faers.db"

conn = sqlite3.connect(db_path)

In [ ]:


tables = pd.read_sql_query(
"""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""",
conn
)

tables

,name
0,demo
1,drug
2,indi
3,outc
4,reac
5,rpsr
6,ther


In [3]:
for table in tables["name"]:
    
    query = f"SELECT COUNT(*) as count FROM {table}"
    result = pd.read_sql_query(query, conn)
    
    print(f"{table}: {result['count'][0]:,}")

demo: 2,422,968
drug: 9,645,822
indi: 5,956,748
outc: 1,478,895
reac: 7,240,344
rpsr: 57,993
ther: 2,822,975


In [4]:
for table in tables["name"]:
    
    query = f"SELECT * FROM {table} LIMIT 5"
    df = pd.read_sql_query(query, conn)
    
    print(f"\n===== {table.upper()} =====")
    print(df.columns)


===== DEMO =====
Index(['primaryid', 'caseid', 'caseversion', 'i_f_code', 'event_dt', 'mfr_dt',
       'init_fda_dt', 'fda_dt', 'rept_cod', 'auth_num', 'mfr_num', 'mfr_sndr',
       'lit_ref', 'age', 'age_cod', 'age_grp', 'sex', 'e_sub', 'wt', 'wt_cod',
       'rept_dt', 'to_mfr', 'occp_cod', 'reporter_country', 'occr_country',
       'source_quarter'],
      dtype='str')

===== DRUG =====
Index(['primaryid', 'caseid', 'drug_seq', 'role_cod', 'drugname', 'prod_ai',
       'val_vbm', 'route', 'dose_vbm', 'cum_dose_chr', 'cum_dose_unit',
       'dechal', 'rechal', 'lot_num', 'exp_dt', 'nda_num', 'dose_amt',
       'dose_unit', 'dose_form', 'dose_freq', 'source_quarter'],
      dtype='str')

===== INDI =====
Index(['primaryid', 'caseid', 'indi_drug_seq', 'indi_pt', 'source_quarter'], dtype='str')

===== OUTC =====
Index(['primaryid', 'caseid', 'outc_cod', 'source_quarter'], dtype='str')

===== REAC =====
Index(['primaryid', 'caseid', 'pt', 'drug_rec_act', 'source_quarter'], dtype='str')


In [5]:
demo = pd.read_sql_query(
"""
SELECT *
FROM demo
LIMIT 20
""",
conn
)

demo

,primaryid,caseid,caseversion,i_f_code,event_dt,mfr_dt,init_fda_dt,fda_dt,rept_cod,auth_num,...,sex,e_sub,wt,wt_cod,rept_dt,to_mfr,occp_cod,reporter_country,occr_country,source_quarter
0,1001678125,10016781,25,F,20120330.0,20240125,20140318,20240129,EXP,None,...,F,Y,NaN,NaN,20240129,None,HP,CA,CA,Q1
1,1002872124,10028721,24,F,20171025.0,20240228,20140321,20240304,EXP,None,...,F,Y,NaN,NaN,20240304,None,MD,CA,CA,Q1
2,100293663,10029366,3,F,NaN,20240114,20140322,20240122,EXP,None,...,M,Y,NaN,NaN,20240122,None,HP,AU,AU,Q1
3,1005450710,10054507,10,F,NaN,20231222,20140402,20240102,PER,None,...,F,Y,NaN,NaN,20240102,None,MD,US,US,Q1
4,1005762118,10057621,18,F,20140204.0,20240129,20140404,20240211,EXP,None,...,M,Y,53.00,KG,20240212,None,CN,CA,CA,Q1
5,1007468611,10074686,11,F,20090714.0,20240207,20140414,20240219,EXP,None,...,M,Y,NaN,NaN,20240219,None,HP,CA,CA,Q1
6,1013616272,10136162,72,F,20120821.0,20240120,20140429,20240130,EXP,None,...,F,Y,112.00,KG,20240130,None,MD,CA,CA,Q1
7,1014222252,10142222,52,F,20131201.0,20240116,20140430,20240118,EXP,None,...,M,Y,89.50,KG,20240118,None,CN,CA,NaN,Q1
8,1014280316,10142803,16,F,20130101.0,20240111,20140430,20240123,EXP,None,...,M,Y,74.00,KG,20240123,None,MD,CA,CA,Q1
9,101539902,10153990,2,F,20140301.0,20240209,20140506,20240212,PER,None,...,F,Y,61.22,KG,20240212,None,MD,US,US,Q1


In [ ]:
#List all column names in all sample outputs

import pandas as pd

tables = ["demo", "drug", "indi", "outc", "reac", "rpsr", "ther"]

for table in tables:
    query = f"SELECT * FROM {table} LIMIT 1"
    df = pd.read_sql_query(query, conn)

    print(f"\n===== {table.upper()} =====")
    print(list(df.columns))


===== DEMO =====
['primaryid', 'caseid', 'caseversion', 'i_f_code', 'event_dt', 'mfr_dt', 'init_fda_dt', 'fda_dt', 'rept_cod', 'auth_num', 'mfr_num', 'mfr_sndr', 'lit_ref', 'age', 'age_cod', 'age_grp', 'sex', 'e_sub', 'wt', 'wt_cod', 'rept_dt', 'to_mfr', 'occp_cod', 'reporter_country', 'occr_country', 'source_quarter']

===== DRUG =====
['primaryid', 'caseid', 'drug_seq', 'role_cod', 'drugname', 'prod_ai', 'val_vbm', 'route', 'dose_vbm', 'cum_dose_chr', 'cum_dose_unit', 'dechal', 'rechal', 'lot_num', 'exp_dt', 'nda_num', 'dose_amt', 'dose_unit', 'dose_form', 'dose_freq', 'source_quarter']

===== INDI =====
['primaryid', 'caseid', 'indi_drug_seq', 'indi_pt', 'source_quarter']

===== OUTC =====
['primaryid', 'caseid', 'outc_cod', 'source_quarter']

===== REAC =====
['primaryid', 'caseid', 'pt', 'drug_rec_act', 'source_quarter']

===== RPSR =====
['primaryid', 'caseid', 'rpsr_cod', 'source_quarter']

===== THER =====
['primaryid', 'caseid', 'dsg_drug_seq', 'start_dt', 'end_dt', 'dur', 'd

In [7]:
from pathlib import Path
import sqlite3
import pandas as pd

project_root = Path.cwd().parents[0]
db_path = project_root / "database" / "faers.db"

conn = sqlite3.connect(db_path)

In [8]:
pd.read_sql_query("SELECT COUNT(*) FROM demo", conn)

,COUNT(*)
0,2422968


In [9]:
pd.read_sql_query("SELECT COUNT(DISTINCT caseid) FROM demo", conn)

,COUNT(DISTINCT caseid)
0,1484350


In [13]:
pd.read_sql_query("SELECT COUNT(*) FROM drug", conn)

,COUNT(*)
0,9645822


In [14]:
pd.read_sql_query("SELECT COUNT(*) FROM reac", conn)

,COUNT(*)
0,7240344


In [15]:
top_drugs = pd.read_sql_query("""
SELECT
    drugname,
    COUNT(*) AS report_count
FROM drug
GROUP BY drugname
ORDER BY report_count DESC
LIMIT 20
""", conn)


top_reactions = pd.read_sql_query("""
SELECT
    pt AS reaction,
    COUNT(*) AS report_count
FROM reac
GROUP BY pt
ORDER BY report_count DESC
LIMIT 20
""", conn)


top_outcomes = pd.read_sql_query("""
SELECT
    outc_cod,
    COUNT(*) AS report_count
FROM outc
GROUP BY outc_cod
ORDER BY report_count DESC
""", conn)

In [ ]:
top_drugs

,outc_cod,report_count
0,OT,806964
1,HO,417316
2,DE,149957
3,LT,62951
4,DS,30682
5,RI,5895
6,CA,5130


In [17]:
top_reactions


,reaction,report_count
0,Off label use,181447
1,Drug ineffective,121957
2,Fatigue,101735
3,Diarrhoea,84657
4,Nausea,81330
5,Product dose omission issue,79926
6,Death,73787
7,Headache,66070
8,Pain,63172
9,Dyspnoea,60074


In [ ]:
top_outcomes

,outc_cod,report_count
0,OT,806964
1,HO,417316
2,DE,149957
3,LT,62951
4,DS,30682
5,RI,5895
6,CA,5130


In [5]:
import pandas as pd
#Set project root variables
from pathlib import Path
import sqlite3
import pandas as pd

project_root = Path.cwd().parents[0]
db_path = project_root / "database" / "faers.db"

conn = sqlite3.connect(db_path)




In [13]:
drug_names = pd.read_sql_query(
"""
SELECT drugname,
       COUNT(*) AS reports,
       CASE
        WHEN COUNT(*) > 100000 THEN 'very_high'
        WHEN COUNT(*) > 50000 THEN 'high'
        ELSE 'moderate'
        END AS report_level
FROM drug
GROUP BY drugname
ORDER BY COUNT(*) DESC
LIMIT 100
""",
conn)

drug_names

,drugname,reports,report_level
0,MOUNJARO,225229,very_high
1,INFLECTRA,182708,very_high
2,DUPIXENT,180993,very_high
3,PREDNISONE,132121,very_high
4,METHOTREXATE,114794,very_high
...,...,...,...
95,LEVOTHYROXINE SODIUM,15667,moderate
96,PAXLOVID,15624,moderate
97,HYDROCHLOROTHIAZIDE,15504,moderate
98,AMLODIPINE BESYLATE,15406,moderate


In [ ]:
top_10_drugs = pd.read_sql_query(
    """
    SELECT drugname, 
        COUNT(*) AS reports, 
        COUNT(*) * 100.0/ (SELECT COUNT(*) FROM drug) AS percent_of_total
    FROM drug
    GROUP BY Drugname
    HAVING COUNT(*) > 1000
    ORDER BY reports DESC
    LIMIT 10
    """,
    conn)

top_10_drugs


,drugname,reports,percent_of_total
0,MOUNJARO,225229,2.334990
1,INFLECTRA,182708,1.894167
2,DUPIXENT,180993,1.876388
3,PREDNISONE,132121,1.369723
4,METHOTREXATE,114794,1.190090
5,VEDOLIZUMAB,90871,0.942076
6,RITUXIMAB,77588,0.804369
7,ACTEMRA,75333,0.780991
8,ACETAMINOPHEN,73673,0.763781
9,ZEPBOUND,59957,0.621585


In [32]:
top_10_A_drugs = pd.read_sql_query(
    """SELECT drugname, COUNT(*) AS reports
    FROM drug
    WHERE drugname LIKE 'A%'
    GROUP BY Drugname
    ORDER BY reports DESC
    Limit 10""",
    conn)

top_10_A_drugs

,drugname,reports
0,ACTEMRA,75333
1,ACETAMINOPHEN,73673
2,ASPIRIN,56004
3,ALBUTEROL SULFATE,39768
4,ATORVASTATIN,37673
5,AMLODIPINE,32339
6,ALBUTEROL,27277
7,ADALIMUMAB,24890
8,AVONEX,21754
9,ALLOPURINOL,17394


In [ ]:
top_10_M_drugs = pd.read_sql_query(
    """SELECT drugname AS top_10_M_drugs, 
        COUNT(*) AS reports
    FROM drug
    WHERE drugname LIKE 'M%'
    GROUP BY Drugname
    ORDER BY reports DESC
    Limit 10""",
    conn
)

top_10_M_drugs


,top_10_M_drugs,reports
0,MOUNJARO,225229
1,METHOTREXATE,114794
2,METFORMIN,30824
3,METHYLPREDNISOLONE,25102
4,MYCOPHENOLATE MOFETIL,24153
5,METHOTREXATE SODIUM,18995
6,METOPROLOL,17726
7,MESALAMINE,16324
8,MAGNESIUM,14213
9,MEPOLIZUMAB,13865


In [41]:
top_10_A_drugs_over_5000 = pd.read_sql_query(
    """ SELECT drugname AS drug, 
        COUNT(*) as reports
        FROM drug
        WHERE drugname LIKE 'A%'
        GROUP BY drugname
        HAVING COUNT(*) > 5000
        ORDER BY reports DESC
        LIMIT 10""",
        
        conn
)

top_10_A_drugs_over_5000

,drug,reports
0,ACTEMRA,75333
1,ACETAMINOPHEN,73673
2,ASPIRIN,56004
3,ALBUTEROL SULFATE,39768
4,ATORVASTATIN,37673
5,AMLODIPINE,32339
6,ALBUTEROL,27277
7,ADALIMUMAB,24890
8,AVONEX,21754
9,ALLOPURINOL,17394


In [ ]:
top_10_IN_drugs_over_5000 = pd.read_sql_query(
    """ SELECT drugname AS drug, 
        COUNT(*) as reports
        FROM drug
        WHERE drugname LIKE '%IN%'
        GROUP BY drugname
        HAVING COUNT(*) > 5000
        ORDER BY reports DESC
        LIMIT 10""",
        
        conn
)

top_10_IN_drugs_over_5000

,drug,reports
0,INFLECTRA,182708
1,ACETAMINOPHEN,73673
2,ASPIRIN,56004
3,SULFASALAZINE,55676
4,GABAPENTIN,38828
5,ATORVASTATIN,37673
6,CETIRIZINE HYDROCHLORIDE,34729
7,HYDROXYCHLOROQUINE,33298
8,AMLODIPINE,32339
9,VITAMIN D3,31795


In [8]:

#### Mounjaro reactions. Table join from drug and reac tables.

import pandas as pd
import sqlite3


mounjaro_reactions = pd.read_sql_query(
"""
SELECT
    r.pt AS reaction,
    COUNT(*) AS reports
FROM drug d
JOIN reac r
    ON d.primaryid = r.primaryid
WHERE d.drugname = 'MOUNJARO'
GROUP BY r.pt
ORDER BY reports DESC
LIMIT 10
""",
conn
)

mounjaro_reactions




,reaction,reports
0,Incorrect dose administered,104752
1,Injection site pain,76615
2,Off label use,59551
3,Nausea,47068
4,Injection site haemorrhage,27952
5,Extra dose administered,24644
6,Injection site erythema,22108
7,Diarrhoea,18419
8,Blood glucose increased,16156
9,Product dose omission issue,15335


In [9]:
top_10_drugs = pd.read_sql_query(
"""
SELECT
    drugname AS drug,
    COUNT(*) AS reports
FROM drug
GROUP BY drugname
ORDER BY reports DESC
LIMIT 10
""",
conn
)

top_10_drugs





top_drug_reactions = pd.read_sql_query(
"""
SELECT
    d.drugname AS drug,
    r.pt AS reaction,
    COUNT(*) AS reports
FROM drug d
JOIN reac r
    ON d.primaryid = r.primaryid
WHERE d.drugname IN (
    SELECT drugname
    FROM drug
    GROUP BY drugname
    ORDER BY COUNT(*) DESC
    LIMIT 10
)
GROUP BY d.drugname, r.pt
ORDER BY d.drugname, reports DESC
""",
conn)

top_drug_reactions

,drug,reaction,reports
0,ACETAMINOPHEN,Off label use,30013
1,ACETAMINOPHEN,Fatigue,16714
2,ACETAMINOPHEN,Pain,15649
3,ACETAMINOPHEN,Headache,15596
4,ACETAMINOPHEN,Nausea,15409
...,...,...,...
39935,ZEPBOUND,Anorexia nervosa,1
39936,ZEPBOUND,Anaphylactic shock,1
39937,ZEPBOUND,Anal fistula,1
39938,ZEPBOUND,Anal abscess,1
